In [ ]:
import os 
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
import pickle
import anndata
from anndata import AnnData
import anndata as ad
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
def pickle_data(data, path):

    with open(path, "wb") as f:
        pickle.dump(data, f)

def load_pickle(path): 

    with open(path, "rb") as f: 
        return pickle.load(f)

### Examine Hartwig RNA

In [ ]:
example_rna_data = pd.read_csv('/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/raw/H00009481.isf.transcript_data.csv')
example_rna_data.head()

In [ ]:
gene_id_to_symbol = dict(zip(example_rna_data['GeneId'], example_rna_data['GeneName']))

In [ ]:
import requests

def get_protein_coding_genes(gene_id_to_symbol, batch_size=1000):
    url = "https://rest.ensembl.org/lookup/id"
    headers = {"Content-Type": "application/json", "Accept": "application/json"}
    
    gene_ids = list(gene_id_to_symbol.keys())
    protein_coding = []
    
    for i in tqdm(range(0, len(gene_ids), batch_size)):
        batch = gene_ids[i:i+batch_size]
        response = requests.post(url, headers=headers, json={"ids": batch})
        data = response.json()
        
        protein_coding += [
            gene_id_to_symbol[gid]
            for gid, info in data.items()
            if isinstance(info, dict) and info.get("biotype") == "protein_coding"
        ]
    
    return protein_coding

In [ ]:
protein_coding_genes = get_protein_coding_genes(gene_id_to_symbol)

In [ ]:
len(protein_coding_genes)

In [ ]:
# in example_rna_data get value in AdjTPM column for gene 'PAX8'
pax8_tpm = example_rna_data[example_rna_data['GeneName'] == 'PAX8']['AdjTPM'].sum()
print(f"PAX8 TPM in H00009481: {pax8_tpm}")

In [ ]:
gene_names = example_rna_data['GeneName'].unique()
len(gene_names)

### Process Hartwig RNA

In [ ]:
def load_tcga_gene_symbols(pkl_path: str) -> list[str]:
    with open(pkl_path, "rb") as f:
        genes = pickle.load(f)
    genes = list(genes)
    if not genes:
        raise ValueError(f"Empty gene list in {pkl_path}")
    return [str(g) for g in genes]


def list_patient_files(raw_dir: str) -> list[Path]:
    files = sorted(Path(raw_dir).glob("*.isf.transcript_data.csv"))
    if not files:
        raise FileNotFoundError(f"No *.isf.transcript_data.csv files found in {raw_dir}")
    return files

def patient_id_from_filename(fp: Path) -> str:
    return fp.name.replace(".isf.transcript_data.csv", "")

In [ ]:
def load_symbol_level_tpm_filtered(
    csv_path: Path,
    tpm_col: str = "AdjTPM",
) -> pd.Series:
    """
    Returns Series indexed by gene SYMBOL, summing TPM across transcripts,
    filtered to allowed_symbols.
    """
    df = pd.read_csv(csv_path, usecols=["GeneName", tpm_col])
    df = df.dropna(subset=["GeneName"]).copy()
    df["GeneName"] = df["GeneName"].astype(str)

    if df.empty:
        return pd.Series(dtype=float)

    sym_tpm = df.groupby("GeneName", sort=False)[tpm_col].sum()
    sym_tpm = pd.to_numeric(sym_tpm, errors="coerce").fillna(0.0)
    return sym_tpm

In [ ]:
def build_hartwig_filtered_anndata_unordered(
    raw_dir: str,
    tpm_col: str = "AdjTPM",
    verbose_every: int = 25,
) -> ad.AnnData:
    files = list_patient_files(raw_dir)

    print(f"Found {len(files)} patients")
    print(f"Using TPM column: {tpm_col}")

    # Pass 1: read each patient, store filtered series, collect union genes
    patient_ids = []
    series_by_patient = []
    gene_union = set()

    for i, fp in tqdm(enumerate(files)):
        pid = patient_id_from_filename(fp)
        patient_ids.append(pid)

        s = load_symbol_level_tpm_filtered(fp, tpm_col=tpm_col)
        series_by_patient.append(s)
        gene_union.update(s.index.tolist())

        if verbose_every and (i + 1) % verbose_every == 0:
            print(f"[{i+1}/{len(files)}] processed; genes matched this sample: {len(s)}")

    gene_list = sorted(gene_union)  # ordering arbitrary
    print(f"Genes retained (TCGA and Hartwig): {len(gene_list)}")

    sym_to_col = {g: j for j, g in enumerate(gene_list)}
    X = np.zeros((len(files), len(gene_list)), dtype=np.float32)

    for i, s in enumerate(series_by_patient):
        row = X[i]
        for sym, val in s.items():
            row[sym_to_col[sym]] = float(val)

    obs = pd.DataFrame({"patient_id": patient_ids}, index=patient_ids)
    var = pd.DataFrame(index=gene_list)
    var["gene_symbol"] = gene_list

    # log_variance on log1p(TPM)
    X_log = np.log1p(X)
    var["log_variance"] = np.var(X_log, axis=0)

    adata = ad.AnnData(X=X, obs=obs, var=var)
    adata.obs_names.name = "patient_id"
    adata.var_names.name = "gene_symbol"

    print(f"Final shape: {adata.shape}")
    return adata

In [ ]:
raw_dir = "/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/raw"

adata = build_hartwig_filtered_anndata_unordered(
    raw_dir=raw_dir,
    tpm_col="AdjTPM",
)

print(adata)

In [ ]:
adata = ad.read_h5ad("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/RNA/Hartwig_bulk_RNA_processed.h5ad")

In [ ]:
# Get gene ordering
hartwig_var = dict(zip(adata.var["gene_symbol"], adata.var["log_variance"]))

# Filter the dictionary to keep only genes in hartwig_genes
hartwig_var = {gene: var for gene, var in hartwig_var.items() if gene in protein_coding_genes}

# Sort the dictionary by value (variance) in descending order
selected_genes_Hartwig = dict(
    sorted(hartwig_var.items(), key=lambda item: item[1], reverse=True)
)

In [ ]:
len(selected_genes_Hartwig)

In [ ]:
pickle_data(selected_genes_Hartwig, "/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/Hartwig_genes_ordered_by_variance.pkl")

### Sanity checks

In [ ]:
print(adata)
print("dtype:", adata.X.dtype)
print("Any NaNs in X?", np.isnan(adata.X).any())
print("Any inf in X?", np.isinf(adata.X).any())

In [ ]:
print("Max value:", adata.X.max())
print("Mean sample sum:", np.mean(np.sum(adata.X, axis=1)))
print("Median sample sum:", np.median(np.sum(adata.X, axis=1)))

In [ ]:
nonzero_per_sample = (adata.X > 0).sum(axis=1)
print("Median nonzero genes per sample:", np.median(nonzero_per_sample))
print("Min:", nonzero_per_sample.min())
print("Max:", nonzero_per_sample.max())

In [ ]:
zero_var = (adata.var["log_variance"] < -20).sum()
print("Genes with near-zero variance:", zero_var)

In [ ]:
raw_var = adata.X.var(axis=0)
print("Genes with zero raw variance:", (raw_var == 0).sum())

In [ ]:
lv = adata.var["log_variance"].values
print("Min:", lv.min())
print("Median:", np.median(lv))
print("Max:", lv.max())

In [ ]:
print("Duplicate gene symbols:",
      adata.var_names.duplicated().sum())

In [ ]:
from scipy.stats import pearsonr

r, _ = pearsonr(adata.X[0], adata.X[1])
print("Sample 0 vs 1 correlation:", r)

### Cleaned way to build clinical labels

In [ ]:
# Load data
treatment_path = "/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/treatment_responses.tsv"
metadata_path = "/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/metadata.tsv"

treatments = pd.read_csv(treatment_path, sep="\t")
metadata = pd.read_csv(metadata_path, sep="\t")

In [ ]:
# Merge biopsy dates into treatment records
df = treatments.merge(metadata[["sampleId", "biopsyDate", "deathDate"]], on="sampleId", how="left")

In [ ]:
df["startDate"] = pd.to_datetime(df["startDate"], errors="coerce")
df["responseDate"] = pd.to_datetime(df["responseDate"], errors="coerce")
df["biopsyDate"] = pd.to_datetime(df["biopsyDate"], errors="coerce")
df["deathDate"] = pd.to_datetime(df["deathDate"], errors="coerce")

In [ ]:
def get_response_dict(df):
    """
    Returns {sampleId: 0 or 1}
    Criteria: 1 if the patient had a PR or CR on the NEXT treatment after biopsy.
    """
    response_dict = {}
    # Only consider rows with a valid clinical response
    valid_evals = df.dropna(subset=["response"]).copy()
    
    # Filter for treatments starting AFTER biopsy
    valid_evals = valid_evals[valid_evals["startDate"] >= valid_evals["biopsyDate"]]

    for sid, group in valid_evals.groupby("sampleId"):
        # Focus only on the first treatment line after biopsy
        first_treatment_date = group["startDate"].min()
        target_treatment = group[group["startDate"] == first_treatment_date]
        
        # 1 if ANY evaluation in this treatment was CR or PR
        is_responder = int(target_treatment["response"].isin(["CR", "PR"]).any())
        response_dict[sid] = is_responder
        
    return response_dict

In [ ]:
def get_ttp_dict(df):
    """
    Returns {sampleId: {'time': days, 'event': 0 or 1}}
    Criteria: 
    - Event (1) if 'PD' occurs.
    - Censored (0) at the last known responseDate if no 'PD' occurs.
    """
    ttp_dict = {}
    # For survival, need at least a startDate and a responseDate
    survival_df = df.dropna(subset=["startDate", "responseDate"]).copy()
    survival_df = survival_df[survival_df["startDate"] >= survival_df["biopsyDate"]]

    for sid, group in survival_df.groupby("sampleId"):
        # Select the first treatment line after biopsy
        first_treatment_date = group["startDate"].min()
        target_treatment = group[group["startDate"] == first_treatment_date].sort_values("responseDate")
        
        # Look for Progression (PD)
        pd_rows = target_treatment[target_treatment["response"] == "PD"]
        
        if not pd_rows.empty:
            # EVENT: First PD date - treatment start
            first_pd_date = pd_rows["responseDate"].iloc[0]
            time = (first_pd_date - first_treatment_date).days
            event = 1
        else:
            # CENSORED: Last known evaluation date - treatment start
            last_eval_date = target_treatment["responseDate"].max()
            time = (last_eval_date - first_treatment_date).days
            event = 0
            
        # Only store if the timing is non-negative
        if time >= 0:
            ttp_dict[sid] = {"time": time, "event": event}
            
    return ttp_dict

In [ ]:
def get_PFS_dict(df):
    """
    Returns {sampleId: {'time': days, 'event': 1 (PD or Death) or 0 (Censored)}}
    """
    pfs_dict = {}
    
    # Filter for treatments starting AFTER biopsy
    # We drop rows with missing startDate as we can't anchor the timeline
    survival_df = df.dropna(subset=["startDate"]).copy()
    survival_df = survival_df[survival_df["startDate"] >= survival_df["biopsyDate"]]

    for sid, group in survival_df.groupby("sampleId"):
        # Select the first treatment line after biopsy
        first_treatment_date = group["startDate"].min()
        target_treatment = group[group["startDate"] == first_treatment_date].sort_values("responseDate")
        
        # Patient-level death date (metadata is constant across the group)
        death_date = target_treatment["deathDate"].iloc[0]
        
        # 1. Look for Progression (PD)
        pd_rows = target_treatment[target_treatment["response"] == "PD"]
        first_pd_date = pd_rows["responseDate"].min() if not pd_rows.empty else pd.NaT
        
        # 2. Compare PD and Death to find the earliest event
        # If both are NaT, event_dates will be empty
        event_candidates = [d for d in [first_pd_date, death_date] if pd.notna(d)]
        
        if event_candidates:
            # EVENT: Use the earliest date among PD and Death
            event_date = min(event_candidates)
            time = (event_date - first_treatment_date).days
            event = 1
        else:
            # CENSORED: No PD and no Death date recorded
            # Use the latest clinical evaluation date as the censoring point
            last_eval_date = target_treatment["responseDate"].max()
            if pd.notna(last_eval_date):
                time = (last_eval_date - first_treatment_date).days
                event = 0
            else:
                # If no responseDate is available, we can't reliably censor
                continue 
            
        if time >= 0:
            pfs_dict[sid] = {"time": time, "event": event}
            
    return pfs_dict

In [ ]:
# Get Labels
responses = get_response_dict(df)
ttp_durations = get_ttp_dict(df)
pfs_durations = get_PFS_dict(df)


print("Sample IDs with response labels:", len(responses))
print("Sample IDs with TTP labels:", len(ttp_durations))
print("Sample IDs with PFS labels:", len(pfs_durations))

In [ ]:
len(set(responses.keys()).intersection(set(pfs_durations.keys())))

In [ ]:
adata = ad.read_h5ad("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/RNA/Hartwig_bulk_RNA_processed_log1p.h5ad")

In [ ]:
rna_samples = set(adata.obs_names)

rna_samples_with_response = rna_samples.intersection(set(responses.keys()))
rna_samples_with_ttp = rna_samples.intersection(set(ttp_durations.keys()))
rna_samples_with_pfs = rna_samples.intersection(set(pfs_durations.keys()))

print("RNA samples with response labels:", len(rna_samples_with_response))
print("RNA samples with TTP labels:", len(rna_samples_with_ttp))
print("RNA samples with PFS labels:", len(rna_samples_with_pfs))

In [ ]:
common_samples = rna_samples_with_response.intersection(rna_samples_with_ttp).intersection(rna_samples_with_pfs)

time_mismatch = 0
event_mismatch = 0
for sid in common_samples:
    if ttp_durations[sid]["time"] != pfs_durations[sid]["time"]:
        time_mismatch += 1
    if ttp_durations[sid]["event"] != pfs_durations[sid]["event"]:
        event_mismatch += 1

print(f"Among {len(common_samples)} common samples, TTP vs PFS time mismatch: {time_mismatch}, event mismatch: {event_mismatch}")

In [ ]:
# Combine into final label df (has response and ttp)
final_labels = []
for sid in responses.keys():
    if sid in pfs_durations:
        final_labels.append({
            "sampleId": sid,
            "response": responses[sid],
            "time": pfs_durations[sid]["time"],
            "event": pfs_durations[sid]["event"]
        })

label_df = pd.DataFrame(final_labels)
print(f"Final label DataFrame shape: {label_df.shape}")

# Double check there are no nan values or duplicate samples
print("Any NaNs in label_df?", label_df.isna().any().any())
print("Duplicate sampleIds in label_df?", label_df["sampleId"].duplicated().any())

label_df.head()

In [ ]:
# Look at overlap of sampleIds between RNA and labels
rna_samples = set(adata.obs_names)
label_samples = set(label_df["sampleId"])
samples_with_rna_and_label = rna_samples.intersection(label_samples)
print("RNA samples:", len(rna_samples))
print(f"Of which have labels:", len(samples_with_rna_and_label))

In [ ]:
# Saving
label_df.to_csv("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/clinical_labels.csv", index=False)
pickle_data(samples_with_rna_and_label, "/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/samples_with_RNA_and_labels.pkl")

In [ ]:
# Check class imbalance
filtered_labels = label_df[label_df["sampleId"].isin(samples_with_rna_and_label)]
print(f"Filtered label_df shape: {filtered_labels.shape}")
response_counts = filtered_labels["response"].value_counts()

print(response_counts)

In [ ]:
# Of the samples with RNA and labels, how many unique patients are there? (some patients have multiple samples)
samples_with_rna_and_label = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/samples_with_RNA_and_labels.pkl")
metadata = pd.read_csv("/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/metadata.tsv", sep="\t")
metadata = metadata[metadata["sampleId"].isin(samples_with_rna_and_label)]
unique_patients = metadata["patientId"].nunique()
print(f"Unique patients with RNA and labels: {unique_patients}")

In [ ]:
# Map sampleId to patientId
sample_to_patient = dict(zip(metadata["sampleId"], metadata["patientId"]))
pickle_data(sample_to_patient, "/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/sample_to_patient_mapping.pkl")

### Get cancer type mapping

In [ ]:
samples_with_rna_and_label = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/samples_with_RNA_and_labels.pkl")
metadata = pd.read_csv("/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/metadata.tsv", sep="\t")

In [ ]:
filtered_metadata = metadata[metadata["sampleId"].isin(samples_with_rna_and_label)].copy()
print(f"Length of filtered metadata: {len(filtered_metadata)}")

In [ ]:
hartwig_to_tcga = {

    # Very clean matches
    "Breast": "BRCA",
    "Colon": "COAD",
    "Rectum": "READ",
    "Lung": "LUAD",  # Hartwig lung is mixed; LUAD is safest single TCGA lung project
    "Skin": "SKCM",
    "Prostate": "PRAD",
    "Pancreas": "PAAD",
    "Ovary": "OV",
    "Bladder": "BLCA",
    "Kidney": "KIRC",  # dominant kidney subtype
    "Liver": "LIHC",
    "Stomach": "STAD",
    "Cervix": "CESC",
    "Thyroid gland": "THCA",
    "Thymus": "THYM",
    "Testis": "TGCT",
    "Adrenal gland": "ACC",
    "Brain/central nervous system": "GBM",

    # GI / hepatobiliary
    "Esophagus/gastroesophageal junction": "ESCA",
    "Biliary tract": "CHOL",
    "Gallbladder": "CHOL",
    "Ampulla of Vater": "CHOL",

    # Urologic
    "Ureter/renal pelvis": "BLCA",  # urothelial lineage
    "Urethra": "BLCA",

    # Gynecologic
    "Endometrium/uterus": "UCEC",
    "Vulva": "CESC",
    "Vagina": "CESC",

    # Head & Neck
    "Oral cavity/tongue": "HNSC",
    "Oropharynx": "HNSC",
    "Larynx": "HNSC",
    "Hypopharynx": "HNSC",
    "Nasopharynx": "HNSC",
    "Nasal cavity": "HNSC",
    "Salivary gland": "HNSC",

    # Other solid
    "Soft tissue": "SARC",
    "Pleura": "MESO",
    "Penis": "CESC",  # closest HPV-driven squamous analog
    "Appendix": "COAD",
    "Anus": "CESC",   # closest HPV-driven squamous analog
}

In [ ]:
# Map Hartwig metadata sampleId column to TCGA project based on primaryTumorLocation
filtered_metadata["TCGA_project"] = filtered_metadata["primaryTumorLocation"].map(hartwig_to_tcga, None)

In [ ]:
# Create dict mapping, keys are only the sampleIds with non None TCGA project, values are TCGA project
sample_to_tcga = dict(zip(filtered_metadata.dropna(subset=["TCGA_project"])["sampleId"], filtered_metadata.dropna(subset=["TCGA_project"])["TCGA_project"]))

In [ ]:
pickle_data(sample_to_tcga, "../../data/Hartwig/id_to_tcga_mapping.pkl")

### Get cohort statistics

In [ ]:
samples_with_rna_and_label = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/samples_with_RNA_and_labels.pkl")
metadata = pd.read_csv("/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/metadata.tsv", sep="\t")

In [ ]:
filtered_metadata = metadata[metadata["sampleId"].isin(samples_with_rna_and_label)].copy()
print(f"Length of filtered metadata: {len(filtered_metadata)}")
filtered_metadata.head()

In [ ]:
filtered_metadata["primaryTumorLocation"].value_counts()

In [ ]:
tumor_location_TCGA_organ_map = {
    # --- Thoracic & Respiratory ---
    "Lung": ["LUAD", "LUSC"],
    "Pleura": ["MESO"],
    "Thymus": ["THYM"],

    # --- Gastrointestinal ---
    "Esophagus/gastroesophageal junction": ["ESCA"],
    "Stomach": ["STAD"],
    "Colon": ["COAD"],
    "Rectum": ["READ"],
    "Liver": ["LIHC"],
    "Biliary tract": ["CHOL"],
    "Pancreas": ["PAAD"],

    # --- Genitourinary ---
    "Bladder": ["BLCA"],
    "Kidney": ["KIRC", "KIRP", "KICH"],
    "Prostate": ["PRAD"],
    "Testis": ["TGCT"],

    # --- Gynecologic ---
    "Breast": ["BRCA"],
    "Ovary": ["OV"],
    "Endometrium/uterus": ["UCEC", "UCS"],
    "Cervix": ["CESC"],

    # --- Head and Neck ---
    "Oral cavity/tongue": ["HNSC"],
    "Oropharynx": ["HNSC"],
    "Larynx": ["HNSC"],
    "Hypopharynx": ["HNSC"],
    "Nasopharynx": ["HNSC"],
    "Nasal cavity": ["HNSC"],

    # --- Endocrine & Others ---
    "Thyroid gland": ["THCA"],
    "Adrenal gland": ["ACC", "PCPG"],
    "Skin": ["SKCM"],
    "Soft tissue": ["SARC"],
    "Brain/central nervous system": ["LGG", "GBM"],
    "Eye": ["UVM"],
    "Lymphoid": ["DLBC"]
}

In [ ]:
sample_to_cancer_type_map = {}

unknown_locations = set()
ambiguous_locations = []
unmapped_locations = []

for sample in samples_with_rna_and_label:
    location = filtered_metadata[filtered_metadata["sampleId"] == sample]["primaryTumorLocation"].iloc[0]
    tcga_types = tumor_location_TCGA_organ_map.get(location, ["Unmapped"])
    
    if location == "Unknown":
        unknown_locations.add(sample)
    elif len(tcga_types) > 1:
        ambiguous_locations.append([sample, location, tcga_types])
    elif tcga_types == ["Unmapped"]:
        unmapped_locations.append([sample, location])
    else:
        sample_to_cancer_type_map[sample] = tcga_types

In [ ]:
# Print summary
print(f"Total samples with RNA and labels: {len(samples_with_rna_and_label)}")
print(f"Samples with mapped tumor location: {len(sample_to_cancer_type_map)}")
print(f"Samples with unknown tumor location: {len(unknown_locations)}")
print(f"Samples with ambiguous tumor location: {len(ambiguous_locations)}")
print(f"Samples with unmapped tumor location: {len(unmapped_locations)}")

In [ ]:
# Make ambiguous locations a dataframe
ambiguous_df = pd.DataFrame(ambiguous_locations, columns=["sampleId", "tumorLocation", "possibleTCGAtypes"])

# Get value counts of tumor location
ambiguous_df["tumorLocation"].value_counts()

In [ ]:
hartwig_tumor_type_to_tcga = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/hartwig_to_tcga_cancer_mapping.pkl")
hartwig_tumor_type_to_tcga

In [ ]:
# Second round: use primaryTumorType mapping
unmapped_samples = set()
newly_mapped_samples = {}
for sample in samples_with_rna_and_label:
    if sample in sample_to_cancer_type_map:
        continue  # already mapped by location
    primary_type = filtered_metadata[filtered_metadata["sampleId"] == sample]["primaryTumorType"].iloc[0]
    tcga_type = hartwig_tumor_type_to_tcga.get(primary_type, None)
    
    if tcga_type is None:
        unmapped_samples.add(sample)
    else:
        newly_mapped_samples[sample] = tcga_type
        
print(f"Newly mapped samples using primaryTumorType: {len(newly_mapped_samples)}")
print(f"Still unmapped samples after primaryTumorType: {len(unmapped_samples)}")

In [ ]:
# Create dataframe for unmapped samples with sampleId, primaryTumorType, and primaryTumorLocation
unmapped_df = pd.DataFrame([
    {
        "sampleId": sample,
        "primaryTumorType": filtered_metadata[filtered_metadata["sampleId"] == sample]["primaryTumorType"].iloc[0],
        "primaryTumorLocation": filtered_metadata[filtered_metadata["sampleId"] == sample]["primaryTumorLocation"].iloc[0]
    }
    for sample in unmapped_samples
])
print(f"Unmapped samples dataframe shape: {unmapped_df.shape}")


In [ ]:
unmapped_df.to_csv("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/samples_with_unmapped_TCGA_cancer_type.csv", index=False)

In [ ]:
# Plot histogram of cancer organ types from filtered metadata

import matplotlib.pyplot as plt
location_counts = filtered_metadata["primaryTumorLocation"].value_counts()
# filter only top 10 locations for better visualization
location_counts = location_counts.head(10)
plt.figure(figsize=(12, 6))
location_counts.plot(kind="bar")
# print the counts on top of the bars
for index, value in enumerate(location_counts):
    plt.text(index, value, str(value), ha="center", va="bottom")
plt.title("Distribution of Primary Tumor Locations in Hartwig Samples with RNA and Labels")
plt.xlabel("Primary Tumor Location")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
lung_metadata_df = filtered_metadata[filtered_metadata["primaryTumorLocation"] == "Lung"]
lung_metadata_df["primaryTumorType"].value_counts()

#### Investigate multiple samples per patient

In [ ]:
# How many unique patientIds in filtered sample metadata?
filtered_metadata["patientId"].value_counts()

In [ ]:
# Print the value counts of patients over 1
len(filtered_metadata["patientId"].value_counts()[filtered_metadata["patientId"].value_counts() > 1])

In [ ]:
# How many samples don't have primaryTumorLocation?
missing_location = filtered_metadata["primaryTumorLocation"].isna().sum()
print(f"Samples missing primaryTumorLocation: {missing_location} out of {len(filtered_metadata)}")

In [ ]:
# Save sample_to_cancer, sample_to_organ 
sample_to_cancer = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/sample_to_cancer_mapping.pkl")
sample_to_organ = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/sample_to_organ_mapping.pkl")

In [ ]:
# Filter dicts
filtered_sample_to_cancer = {sid: sample_to_cancer[sid] if sid in sample_to_cancer else "Unknown" for sid in samples_with_rna_and_label}
filtered_sample_to_organ = {sid: sample_to_organ[sid] if sid in sample_to_organ else "Unknown" for sid in samples_with_rna_and_label}

In [ ]:
len(set(filtered_metadata["primaryTumorType"].values))

In [ ]:
print(f"Number of cancer types in filtered data: {len(set(filtered_sample_to_cancer.values()))}")
print(f"Number of organs in filtered data: {len(set(filtered_sample_to_organ.values()))}")

In [ ]:
# Drug information

# Load the treatment data
treatment_path = '/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/treatment_responses.tsv'
df_treat = pd.read_csv(treatment_path, sep='\t')

# 1. Filter using your existing sampleIds set
# Ensure we only look at rows matching your RNA-eligible samples
df_filtered = df_treat[df_treat['sampleId'].isin(adata.obs_names)].copy()


# 2. Calculate Comprehensive Stats
stats = {
    "Samples with Treatment Info": df_filtered['sampleId'].nunique(),
    "Unique Patients": df_filtered['patientId'].nunique(),
    "Unique Drug Names": df_filtered['name'].nunique(),
    "Unique Drug Types": df_filtered['type'].nunique(),
    "Unique Mechanisms": df_filtered['mechanism'].nunique(),
    "Total Treatment Records": len(df_filtered)
}

# 3. Display Results
print("--- Hartwig Cohort Treatment Stats ---")
for k, v in stats.items():
    print(f"{k:30}: {v}")

print("\n--- Top 5 Drug Types ---")
print(df_filtered['type'].value_counts().head(5))

print("\n--- Response Summary (per treatment) ---")
print(df_filtered['response'].value_counts(dropna=False))

# Optional: Check for samples missing treatment info
missing_count = len(samples_with_rna_and_label) - df_filtered['sampleId'].nunique()
if missing_count > 0:
    print(f"\nNote: {missing_count} samples from your set have no entries in the treatment TSV.")

In [ ]:
len(adata)

In [ ]:
# Plot histogram of cancer types, sort by count and show top 10. only show non "Unknown"
import matplotlib.pyplot as plt
cancer_counts = pd.Series(filtered_sample_to_cancer).value_counts()
filtered_cancer_counts = cancer_counts[cancer_counts.index != "Unknown"].sort_values(ascending=False)
unknown_count = cancer_counts.get("Unknown", 0)
plt.figure(figsize=(10, 6))
filtered_cancer_counts.head(5).plot(kind="bar")
plt.title("Top 5 Cancer Types (mapped to TCGA codes)")
# put unknown count on plot top right
plt.text(0.95, 0.95, f"Unknown: {unknown_count}", transform=plt.gca().transAxes, ha="right", va="top", fontsize=12)
plt.xlabel("Cancer Type")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0, fontsize=15)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
cancer_counts = Counter(filtered_sample_to_cancer.values())
plt.figure(figsize=(12, 6))
plt.bar(cancer_counts.keys(), cancer_counts.values())
plt.xticks()
plt.title("Distribution of Cancer Types in Filtered Hartwig Data")
plt.xlabel("Cancer Type")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()

### Get cancer types from metadata

In [ ]:
cancer_mapping = {
    "ACC":0,"BLCA":1,"BRCA":2,"CESC":3,"CHOL":4,"COAD":5,"DLBC":6,"ESCA":7,
    "GBM":8,"HNSC":9,"KICH":10,"KIRC":11,"KIRP":12,"LAML":13,"LGG":14,
    "LIHC":15,"LUAD":16,"LUSC":17,"MESO":18,"OV":19,"PAAD":20,"PCPG":21,
    "PRAD":22,"READ":23,"SARC":24,"SKCM":25,"STAD":26,"TGCT":27,"THCA":28,
    "THYM":29,"UCEC":30,"UCS":31,"UVM":32
}

hartwig_to_tcga = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/hartwig_to_tcga_cancer_mapping.pkl")

In [ ]:
def build_cancer_and_organ_dicts(metadata_path):

    print("Loading:", metadata_path)
    df = pd.read_csv(metadata_path, sep="\t")

    print("Total rows:", len(df))
    print("Unique sampleIds:", df["sampleId"].nunique())

    # Drop rows without sampleId
    df = df.dropna(subset=["sampleId"])

    sample_to_cancer = {}
    sample_to_organ = {}

    unmapped = set()

    for _, row in df.iterrows():
        sid = row["sampleId"]
        tumor_type = row["primaryTumorType"]
        organ = row["primaryTumorLocation"]

        # Map to TCGA csncer type code
        tcga_code = hartwig_to_tcga.get(tumor_type)

        if tcga_code is None:
            unmapped.add(tumor_type)
        else:
            sample_to_cancer[sid] = tcga_code

        # Organ mapping (just raw string)
        if pd.notna(organ):
            sample_to_organ[sid] = organ

    print("Samples mapped to TCGA cancer codes:", len(sample_to_cancer))
    print("Samples mapped to organ:", len(sample_to_organ))

    if len(unmapped) > 0:
        print("\nUnmapped tumor types:")
        for t in unmapped:
            print("-", t)

    return sample_to_cancer, sample_to_organ


In [ ]:
metadata_path = "/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/metadata.tsv"

sample_to_cancer, sample_to_organ = build_cancer_and_organ_dicts(metadata_path)

print("\nExample cancer mappings:")
print(list(sample_to_cancer.items())[:5])

print("\nExample organ mappings:")
print(list(sample_to_organ.items())[:5])

In [ ]:
patients_without_cancer_type = set()
for sid in hartwig_patients:
    if sid not in sample_to_cancer:
        patients_without_cancer_type.add(sid)
len(patients_without_cancer_type)

In [ ]:
clinical_df[clinical_df["sampleId"] == 'H00000026']

In [ ]:
metadata_df = pd.read_csv("/data/rbg/shared/datasets/Hartwig/isofox_tpm_update_4/metadata.tsv", sep="\t")

In [ ]:
missing_cancers = {}
for sid in patients_without_cancer_type:
    cancer = metadata_df[metadata_df["sampleId"] == sid]["primaryTumorType"].iloc[0]
    if cancer in missing_cancers:
        missing_cancers[cancer] += 1
    else:
        missing_cancers[cancer] = 1
missing_cancers



In [ ]:
# Save sample_to_cancer, sample_to_organ 
pickle_data(sample_to_cancer, "/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/sample_to_cancer_mapping.pkl")
pickle_data(sample_to_organ, "/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/Hartwig/sample_to_organ_mapping.pkl")

### Analyze processed Hartwig embeddings

In [ ]:
embeddings = load_pickle("/data/rbg/users/ashyamal/RNA-FM-benchmarking/data/RNA_embeddings/Hartwig_processed.pkl")

In [ ]:
list(embeddings.keys())[:5]

In [ ]:
embeddings["H00009590"].keys()

In [ ]:
time_count, event_count, response_count, treatment_count = 0, 0, 0, 0
for sid in embeddings.keys():
    emb = embeddings[sid]
    time, event, response, treatment = emb["PFS_time"], emb["PFS_event"], emb["response"], emb["treatment"]
    if time is not None:
        time_count += 1
    if event is not None:
        event_count += 1
    if response is not None:
        response_count += 1
    if treatment is not None:
        treatment_count += 1
print(f"Samples with time: {time_count}")
print(f"Samples with event: {event_count}")
print(f"Samples with response: {response_count}")
print(f"Samples with treatment: {treatment_count}")

### Examine splits

In [ ]:
example_split = load_pickle("../../splits/Hartwig/split_0.pkl")

In [ ]:
train_keys, val_keys, test_keys = example_split['train'], example_split['val'], example_split['test']

In [ ]:
assert len(set(train_keys).intersection(set(val_keys))) == 0
assert len(set(train_keys).intersection(set(test_keys))) == 0
assert len(set(val_keys).intersection(set(test_keys))) == 0